In [1]:
import pandas as pd
import math
from sklearn.metrics import mean_squared_error
import random
from datetime import datetime as dt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, LSTM
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

# Fix random seeds so that re-running the notebook gives comparable numbers
import tensorflow as tf
random.seed(42); np.random.seed(42); tf.random.set_seed(42)


I0000 00:00:1795702182.399230     155 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795702194.425640     155 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1795702201.298425     155 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


### For NIFTY-50 index fund


In [2]:
import os
stock_ticker = '^NSEI'
DATA_FILE = 'nifty50.csv'   # cached copy of the Yahoo Finance download, so results can be reproduced

if os.path.exists(DATA_FILE):
    raw = pd.read_csv(DATA_FILE, parse_dates=['Date'])
else:
    raw = yf.download(stock_ticker, start='2015-01-01', end='2025-07-31', multi_level_index=False).reset_index()
    raw.to_csv(DATA_FILE, index=False)

df = raw[['Date', 'Close']].copy()
df.rename(columns={'Date':'date', 'Close':'close'}, inplace=True)


In [3]:
df2 = pd.DataFrame(np.log(df['close']))
df2['date'] = pd.to_datetime(df['date']).dt.date

df2.set_index('date', inplace=True)
df2.rename(columns={'^NSEI':'close'}, inplace=True)
df2

,close
date,
2015-01-02,9.035445
2015-01-05,9.033412
2015-01-06,9.002990
2015-01-07,8.999879
2015-01-08,9.016100
...,...
2025-07-25,10.120090
2025-07-28,10.113785
2025-07-29,10.119449


In [4]:
plt.figure(figsize=(20,10))
plt.plot(df2['close'])
plt.title('Closed', fontsize=20)
plt.ylabel('Price In Rupees', fontsize=16)
plt.xlabel('Date', fontsize=16)
plt.show()

### ADF Test

In [5]:
def check_stationary(data):
    result = adfuller(data)
    print('ADF Statistic:', result[0])
    print('p-value:', result[1])
    if result[1] <= 0.05:
        print('The data is stationary.')
    else:
        print('The data is not stationary.')

check_stationary(df2.close)

first_diffs = df2.close - df2.close.shift(1)

df2['FirstDifference'] = first_diffs
df2.dropna(inplace=True)

check_stationary(df2.FirstDifference)

ADF Statistic: -0.1989865789027057
p-value: 0.9386423067630772
The data is not stationary.
ADF Statistic: -14.131475356234153
p-value: 2.3295769617788547e-26
The data is stationary.


### KPSS Test

In [6]:
from statsmodels.tsa.stattools import kpss
import pandas as pd

def kpss_test(series, **kw):

    statistic, p_value, n_lags, critical_values = kpss(series, **kw)

    # Format and print the results
    print(f'KPSS Statistic: {statistic}')
    print(f'P-Value: {p_value}')
    print(f'Number of Lags: {n_lags}')
    print('Critical Values:')
    for key, value in critical_values.items():
        print(f'   {key}: {value}')

    # Interpret the results
    if p_value < 0.05:
        print("\nConclusion: The series is NOT stationary (reject H0).")
    else:
        print("\nConclusion: The series IS stationary (fail to reject H0).")



In [7]:
print("--- KPSS Test on Original Log-Close Price ---")
kpss_test(df2['close'])

print("\n" + "="*40 + "\n")

# Test the first-differenced data
print("--- KPSS Test on First Difference ---")
kpss_test(df2['FirstDifference'].dropna())

--- KPSS Test on Original Log-Close Price ---
KPSS Statistic: 8.074364992249299
P-Value: 0.01
Number of Lags: 30
Critical Values:
   10%: 0.347
   5%: 0.463
   2.5%: 0.574
   1%: 0.739

Conclusion: The series is NOT stationary (reject H0).


--- KPSS Test on First Difference ---
KPSS Statistic: 0.06827596760415806
P-Value: 0.1
Number of Lags: 12
Critical Values:
   10%: 0.347
   5%: 0.463
   2.5%: 0.574
   1%: 0.739

Conclusion: The series IS stationary (fail to reject H0).


In [8]:
plt.figure(figsize=(20,10))
plt.plot(df2.FirstDifference)
plt.title('Closed', fontsize=20)
plt.ylabel('Price in Rupees', fontsize=16)
plt.xlabel('Date', fontsize=16)
plt.show()

In [9]:
# Plotting ACF plot to choose value for MA

plot_acf(df2.FirstDifference)
plt.show()

# Plotting PACF plot to choose value for AR

plot_pacf(df2.FirstDifference)
plt.show()

### ARIMA Model

In [10]:
# Creating the ARIMA model and fitting the values

model = ARIMA(df2.FirstDifference.dropna(), order=(1,1,0))
result = model.fit()

fvalues = result.fittedvalues
result.summary()

residual = result.resid
residual = residual.cumsum()

In [11]:

a = df2.close - df2.FirstDifference
diff_results = residual + residual.shift(-1)
diff_results = diff_results + a

In [12]:
df1 = diff_results
df1.dropna(inplace=True)

# Normalizing the data

scaler = MinMaxScaler(feature_range=(0,1))
df1 = scaler.fit_transform(np.array(df1).reshape(-1,1))

In [13]:
# Defining the train and test data size

training_size=int(len(df1)*0.70)
test_size=len(df1)-training_size
train_data,test_data=df1[:training_size],df1[training_size:]

In [14]:
def create_dataset(dataset, time_step=60):
	dataX, dataY = [], []
	for i in range(len(dataset)-time_step-1):
		a = dataset[i:(i+time_step)]
		dataX.append(a)
		dataY.append(dataset[i + time_step])
	return np.array(dataX), np.array(dataY)

time_step = 60
X_train, y_train = create_dataset(train_data, time_step)
X_test, y_test = create_dataset(test_data, time_step)

In [15]:
X_train =X_train.reshape(X_train.shape[0],X_train.shape[1] , 1)
X_test = X_test.reshape(X_test.shape[0],X_test.shape[1] , 1)

len(X_train), len(y_train), len(X_test), len(y_test)

(1761, 1761, 720, 720)

### LSTM Model

In [16]:
# Creating LSTM model

model=Sequential()
model.add(Input(shape=(X_train.shape[1],1)))
model.add(LSTM(64))
model.add(Dropout(0.3))
model.add(Dense(25))
model.add(Dense(1))
model.compile(loss='mean_squared_error',optimizer='adam', metrics=['accuracy'])

In [17]:
# Summary for LSTM model

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 25)             │         1,625 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            26 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,547 (72.45 KB)

 Trainable params: 18,547 (72.45 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=5,batch_size=64,verbose=1)

Epoch 1/5


 1/28 ━━━━━━━━━━━━━━━━━━━━ 40s 1s/step - accuracy: 0.0000e+00 - loss: 0.1387

 5/28 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.0000e+00 - loss: 0.0753

 8/28 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0000e+00 - loss: 0.0511

12/28 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0000e+00 - loss: 0.0403

16/28 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0000e+00 - loss: 0.0321

... (40 outputs left out)

19/28 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 8.2237e-04 - loss: 0.0017

23/28 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 6.7935e-04 - loss: 0.0017

27/28 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 5.7870e-04 - loss: 0.0017

28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 5.6786e-04 - loss: 0.0017 - val_accuracy: 0.0014 - val_loss: 5.5268e-04


In [19]:
train_predict=model.predict(X_train)
test_predict=model.predict(X_test)

 1/56 ━━━━━━━━━━━━━━━━━━━━ 6s 119ms/step

13/56 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step  

24/56 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

36/56 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

48/56 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step

56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


 1/23 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step

11/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 

21/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step

23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


In [20]:
mape_stock = mean_absolute_percentage_error(y_test,test_predict)
rmse_stock = np.sqrt(mean_squared_error(y_test,test_predict))
mase_stock = np.mean(np.abs(y_test - test_predict)) / np.mean(np.abs(pd.DataFrame(y_test).diff().dropna()))
print(f"NIFTY - MAPE: {mape_stock}, RMSE: {rmse_stock}, MASE: {mase_stock}")

NIFTY - MAPE: 0.024803814592637898, RMSE: 0.023509218795782664, MASE: 3.8673805058358894


In [21]:
final_test = pd.DataFrame(df2.iloc[training_size + time_step : training_size + time_step + len(y_test)])
final_test['close'] = np.exp(final_test['close'])
final_test['test'] = np.exp(scaler.inverse_transform(test_predict))

In [22]:
# Plotting the predictied price wrt original price of test

plt.figure(figsize=(15,8))
plt.plot(final_test.close, label='Original Price', color='red')
plt.plot(final_test.test, label='Predicted Price', color='green')
plt.title('NIFTY - LSTM', fontweight='bold',)
plt.ylabel('Price in Rupees', fontweight='bold')
plt.xlabel('Date', fontweight='bold')
plt.legend()
plt.show()

### Bi-LSTM Model

In [23]:
# Build BiLSTM model
from tensorflow.keras.layers import Bidirectional, LSTM, Dense, Dropout

model_bilstm = Sequential([
    Input(shape=(X_train.shape[1],1)),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.3),
    Dense(32,activation='tanh'),
    Dropout(0.3),
    Dense(1)  # Binary classification
])

# Compile the model
model_bilstm.compile(optimizer='adam', loss='mse')

model_bilstm.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 128)            │        33,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 37,953 (148.25 KB)

 Trainable params: 37,953 (148.25 KB)

 Non-trainable params: 0 (0.00 B)

In [24]:
model_bilstm.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=5,batch_size=64,verbose=1)

Epoch 1/5


 1/28 ━━━━━━━━━━━━━━━━━━━━ 53s 2s/step - loss: 0.1646

 4/28 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0968

 7/28 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0626

10/28 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0521

13/28 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0440

... (49 outputs left out)

21/28 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0044

24/28 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0046

27/28 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.0046

28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - loss: 0.0046 - val_loss: 7.8029e-04


In [25]:
train_predict_bi=model_bilstm.predict(X_train)
test_predict_bi=model_bilstm.predict(X_test)

 1/56 ━━━━━━━━━━━━━━━━━━━━ 12s 231ms/step

10/56 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step   

21/56 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step

31/56 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step

42/56 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step

50/56 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step

... (2 outputs left out)

 1/23 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step

11/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 

21/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step

23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


In [26]:
mape_stock = mean_absolute_percentage_error(y_test,test_predict_bi)
rmse_stock = np.sqrt(mean_squared_error(y_test,test_predict_bi))
mase_stock = np.mean(np.abs(y_test - test_predict_bi)) / np.mean(np.abs(pd.DataFrame(y_test).diff().dropna()))
print(f"NIFTY - MAPE: {mape_stock}, RMSE: {rmse_stock}, MASE: {mase_stock}")

NIFTY - MAPE: 0.028595155073505445, RMSE: 0.027933603825588876, MASE: 4.5357670591934225


In [27]:
final_test = pd.DataFrame(df2.iloc[training_size + time_step : training_size + time_step + len(y_test)])
final_test['close'] = np.exp(final_test['close'])
final_test['test'] = np.exp(scaler.inverse_transform(test_predict_bi))

In [28]:
# Plotting the predictied price wrt original price of test

plt.figure(figsize=(15,8))
plt.plot(final_test.close, label='Original Price', color='red')
plt.plot(final_test.test, label='Predicted Price', color='green')
plt.title('NIFTY - Bi-LSTM', fontweight='bold',)
plt.ylabel('Price in Rupees', fontweight='bold')
plt.xlabel('Date', fontweight='bold')
plt.legend()
plt.show()

### GRU Model

In [29]:
#Create GRU Model

from tensorflow.keras.layers import GRU, Dense, Dropout
model_gru = Sequential([
    Input(shape=(X_train.shape[1],1)),
    GRU(64, return_sequences=False),
    Dropout(0.3),
    Dense(32, activation='tanh'),
    Dense(1)  # single value prediction
])

model_gru.compile(optimizer='adam', loss='mse')
model_gru.summary()
model_gru.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=5,batch_size=64,verbose=1)
train_predict_gru=model_gru.predict(X_train)
test_predict_gru=model_gru.predict(X_test)

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 64)             │        12,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,977 (58.50 KB)

 Trainable params: 14,977 (58.50 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5


... (68 outputs left out)

 1/23 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step

10/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 

19/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step

23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


In [30]:
mape_stock = mean_absolute_percentage_error(y_test,test_predict_gru)
rmse_stock = np.sqrt(mean_squared_error(y_test,test_predict_gru))
mase_stock = np.mean(np.abs(y_test - test_predict_gru)) / np.mean(np.abs(pd.DataFrame(y_test).diff().dropna()))
print(f"NIFTY - MAPE: {mape_stock}, RMSE: {rmse_stock}, MASE: {mase_stock}")

NIFTY - MAPE: 0.01206156428636919, RMSE: 0.012744491109669626, MASE: 1.930034601177273


In [31]:
final_test = pd.DataFrame(df2.iloc[training_size + time_step : training_size + time_step + len(y_test)])
final_test['close'] = np.exp(final_test['close'])
final_test['test'] = np.exp(scaler.inverse_transform(test_predict_gru))

In [32]:
# Plotting the predictied price wrt original price of test

plt.figure(figsize=(15,8))
plt.plot(final_test.close, label='Original Price', color='red')
plt.plot(final_test.test, label='Predicted Price', color='green')
plt.title('NIFTY - GRU', fontweight='bold',)
plt.ylabel('Price in Rupees', fontweight='bold')
plt.xlabel('Date', fontweight='bold')
plt.legend()
plt.show()

### Deep GRU Model

In [33]:
#Deep GRU Architecture

model_dgru = Sequential([
    Input(shape=(X_train.shape[1],1)),
    GRU(64, return_sequences=True),  # 1st GRU layer
    Dropout(0.3),
    GRU(64, return_sequences=True),  # 2nd GRU layer
    Dropout(0.3),
    GRU(32, return_sequences=False),  # 3rd GRU layer (last RNN)
    Dropout(0.2),
    Dense(32, activation='tanh'),
    Dense(1)  # single value prediction
])

model_dgru.compile(optimizer='adam', loss='mse')
model_dgru.summary()

model_dgru.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=5,batch_size=64,verbose=1)
train_predict_dgru=model_dgru.predict(X_train)
test_predict_dgru=model_dgru.predict(X_test)

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_1 (GRU)                     │ (None, 60, 64)         │        12,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 60, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, 60, 64)         │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 60, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ (None, 32)             │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 48,321 (188.75 KB)

 Trainable params: 48,321 (188.75 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5


... (163 outputs left out)

10/23 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step

15/23 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step

20/23 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step

23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step


In [34]:
# Metrics
mape_stock = mean_absolute_percentage_error(y_test, test_predict_dgru)
mse_stock = mean_squared_error(y_test, test_predict_dgru)
rmse_stock = np.sqrt(mse_stock)

# Fixing MASE calculation using y_train (naive forecast based on previous value)
naive_forecast_error = np.mean(np.abs(np.diff(y_train.reshape(-1))))  # If y_train is a NumPy array
# If y_train is a Pandas Series or DataFrame, use: y_train.values.reshape(-1)

mase_stock = np.mean(np.abs(y_test - test_predict_dgru)) / naive_forecast_error

# Print results
print(f"Stock - MAPE: {mape_stock:.4f}, MSE: {mse_stock:.4f}, RMSE: {rmse_stock:.4f}, MASE: {mase_stock:.4f}")

Stock - MAPE: 0.0526, MSE: 0.0022, RMSE: 0.0473, MASE: 6.2175


In [35]:
final_test = pd.DataFrame(df2.iloc[training_size + time_step : training_size + time_step + len(y_test)])
final_test['close'] = np.exp(final_test['close'])
final_test['test'] = np.exp(scaler.inverse_transform(test_predict_dgru))

In [36]:
# Plotting the predictied price wrt original price of test

plt.figure(figsize=(15,8))
plt.plot(final_test.close, label='Original Price', color='red')
plt.plot(final_test.test, label='Predicted Price', color='green')
plt.title('NIFTY - Deep GRU', fontweight='bold',)
plt.ylabel('Price in Rupees', fontweight='bold')
plt.xlabel('Date', fontweight='bold')
plt.legend()
plt.show()

## Evaluation in index points, against baselines

The MAPE / RMSE / MASE printed above are computed on the **MinMax-scaled** series, so they are not errors in index points, and a baseline is needed to say whether a model adds anything.
This section converts every forecast back to NIFTY-50 closing prices, aligns it with the actual close on the same date, and compares it with:

- **Naive**: tomorrow's close = today's close.
- **ARIMA(1,1,0) only**: a one-step-ahead forecast of the log close from an ARIMA fitted on the training period only.

`MASE` here is the model's mean absolute error divided by the in-sample mean absolute one-day change in the training period (the naive one-step error), so a value above 1 means worse than the naive forecast. `Dir. acc.` is the share of days on which the forecast moved in the same direction as the actual close.

In [37]:
# Convert each model's test predictions back to closing prices (index points) and evaluate on the same dates
from sklearn.metrics import mean_absolute_error

dates = diff_results.index[training_size + time_step : training_size + time_step + len(y_test)]
log_close = df2['close']
actual = np.exp(log_close.reindex(dates).values)
prev_close = np.exp(log_close.shift(1).reindex(dates).values)

# in-sample naive one-step error (training period) for MASE, in index points
train_dates = diff_results.index[:training_size]
train_close = np.exp(log_close.reindex(train_dates).values)
naive_train_mae = np.mean(np.abs(np.diff(train_close)))

# ARIMA(1,1,0)-only baseline: fitted on the training period, then applied to the full series for one-step-ahead forecasts
lc = log_close.reset_index(drop=True)
n_train = lc.index[log_close.index.get_indexer(train_dates)].max() + 1
arima_fit = ARIMA(lc.iloc[:n_train], order=(1, 1, 0)).fit()
arima_full = arima_fit.apply(lc)
arima_one_step = np.exp(arima_full.fittedvalues.reindex(log_close.index.get_indexer(dates)).values)

forecasts = {
    'Naive (previous close)': prev_close,
    'ARIMA(1,1,0) only': arima_one_step,
    'ARIMA-LSTM': np.exp(scaler.inverse_transform(test_predict)).ravel(),
    'ARIMA-Bi-LSTM': np.exp(scaler.inverse_transform(test_predict_bi)).ravel(),
    'ARIMA-GRU': np.exp(scaler.inverse_transform(test_predict_gru)).ravel(),
    'ARIMA-Deep GRU': np.exp(scaler.inverse_transform(test_predict_dgru)).ravel(),
}

rows = []
for name, f in forecasts.items():
    err = actual - f
    mae = np.mean(np.abs(err))
    rows.append({
        'Model': name,
        'MAE (index pts)': mae,
        'RMSE (index pts)': np.sqrt(np.mean(err ** 2)),
        'MAPE (%)': 100 * np.mean(np.abs(err) / actual),
        'MASE': mae / naive_train_mae,
        'Dir. acc. (%)': np.nan if name.startswith('Naive') else 100 * np.mean(np.sign(f - prev_close) == np.sign(actual - prev_close)),
    })
results_price = pd.DataFrame(rows).set_index('Model').round(3)
print(f"Test period: {dates[0]} to {dates[-1]}  ({len(dates)} trading days)")
results_price


Test period: 2022-08-30 to 2025-07-29  (720 trading days)


,MAE (index pts),RMSE (index pts),MAPE (%),MASE,Dir. acc. (%)
Model,,,,,
Naive (previous close),124.185,174.792,0.580,1.478,NaN
"ARIMA(1,1,0) only",124.341,174.582,0.581,1.480,45.972
ARIMA-LSTM,495.263,545.239,2.321,5.894,46.111
ARIMA-Bi-LSTM,609.062,716.896,2.791,7.248,45.000
ARIMA-GRU,188.622,239.149,0.831,2.245,57.917
ARIMA-Deep GRU,1220.122,1301.300,5.555,14.520,45.694
